# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:
# Initialization

load_dotenv(override=True)
gemini_api_key=os.getenv('GOOGLE_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key=os.getenv('OPENROUTER_API_KEY')
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url="https://openrouter.ai/api/v1"
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gemini-3.5-flash"
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-or-v1


In [4]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openrouter.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [5]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [6]:
get_ticket_price("Berlin")

Tool called for city Berlin


'The price of a ticket to Berlin is $499'

In [7]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [ ]:
tools

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response =openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        for message in messages:
            print(message)
        response = openrouter.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content


In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [8]:
import sqlite3


In [9]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [10]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [11]:
get_ticket_price("sydney")

DATABASE TOOL CALLED: Getting price for sydney


'No price data available for this city'

In [12]:
def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
        # Query to verify
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} has been set to ${result[0]}" if result else "Failed to set price"

In [13]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

DATABASE TOOL CALLED: Setting price for london
DATABASE TOOL CALLED: Setting price for paris
DATABASE TOOL CALLED: Setting price for tokyo
DATABASE TOOL CALLED: Setting price for sydney


In [14]:
set_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the price of a return ticket for a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city to set the ticket price for",
            },
            "price": {
                "type": "number",
                "description": "The ticket price in dollars",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [15]:
tools = [{"type": "function", "function": price_function},
         {"type": "function", "function": set_price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'set_ticket_price',
   'description': 'Set or update the price of a return ticket for a destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city to set the ticket price for'},
     'price': {'type': 'number',
      'description': 'The ticket price in dollars'}},
    'required': ['destination_city', 'price'],
    'additionalProperties': False}}}]

In [ ]:
set_ticket_price("Tokyo",1200)

In [16]:
import sqlite3

def show_all_prices():
    """Display all prices in the database"""
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT city, price FROM prices')
        results = cursor.fetchall()
        print("Current Prices in Database:")
        for city, price in results:
            print(f"  {city.capitalize()}: ${price}")

show_all_prices()

Current Prices in Database:
  London: $799.0
  Paris: $899.0
  Tokyo: $1420.0
  Sydney: $2999.0


In [17]:
# def handle_tool_calls(message):
#     responses = []
#     for tool_call in message.tool_calls:
#         if tool_call.function.name == "get_ticket_price":
#             arguments = json.loads(tool_call.function.arguments)
#             city = arguments.get('destination_city')
#             price_details = get_ticket_price(city)
#             responses.append({
#                 "role": "tool",
#                 "content": price_details,
#                 "tool_call_id": tool_call.id
#             })
#     return responses




import json

# 1. Map API tool names directly to Python functions
TOOL_MAP = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price,
    # "get_weather": get_weather,  <-- Easily add more tools here later!
}

def handle_tool_call(tool_call):
    #Processes a single tool call.
    fn_name = tool_call.function.name
    
    if fn_name not in TOOL_MAP:
        return {
            "role": "tool",
            "content": f"Error: Tool '{fn_name}' is not supported.",
            "tool_call_id": tool_call.id,
        }

    # Extract arguments safely
    args = json.loads(tool_call.function.arguments)
    
    # Execute function using **kwargs unpack
    result = TOOL_MAP[fn_name](**args)
    
    return {
        "role": "tool",
        "content": str(result),
        "tool_call_id": tool_call.id,
    }

def handle_tool_calls(message):
    #Processes all tool calls in parallel using a list comprehension.
    return [handle_tool_call(call) for call in (message.tool_calls or [])]
    
    

In [18]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openrouter.chat.completions.create(model=MODEL, messages=messages,max_completion_tokens=500, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)

        for m in messages:
            print(m)
        
        response = openrouter.chat.completions.create(model=MODEL, messages=messages,max_completion_tokens=500)

        for r in response.choices[0].message:
            print(r)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>